# LayoutLMv1-Tiny-GST

Commercial-friendly, offline Android target. ML Kit supplies OCR words and boxes; this
student predicts GST fields. The full teacher is used only during training.

**Important:** six PDFs are a smoke-test/error-analysis set, not a general training set.
Upload a labeled JSONL dataset with `words`, `boxes`, and `labels` before full training.


In [ ]:
!pip -q install transformers datasets seqeval onnx onnxruntime onnxruntime-tools
import torch, transformers
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE = Path('/content/drive/MyDrive/gst-engine')
DATA = BASE / 'layoutlm_tiny_labeled'
OUT = BASE / 'models/layoutlmv1-tiny-gst'
print(DATA, OUT)
print('dataset exists:', DATA.exists())


## Dataset contract

Each JSONL record represents one page after ML Kit OCR:

```json
{"image_id":"x/page-1", "words":["GSTIN", "27..."], "boxes":[[10,20,80,40], ...], "labels":["O", "B-SELLER_GSTIN"]}
```

Boxes are normalized to 0..1000. Labels use BIOES-style tags. Split by vendor/layout family,
not random pages from the same invoice.


In [ ]:
import json
from collections import Counter
train_path = DATA/'train.jsonl'
valid_path = DATA/'valid.jsonl'
assert train_path.exists() and valid_path.exists(), 'Create labeled train.jsonl and valid.jsonl first'
for p in [train_path, valid_path]:
    rows=[json.loads(x) for x in p.read_text().splitlines() if x.strip()]
    print(p.name, len(rows), 'pages', 'labels', Counter(y for r in rows for y in r['labels']).most_common(8))


In [ ]:
# Student configuration: 6 layers x 256 hidden x 4 heads.
from transformers import LayoutLMConfig, LayoutLMForTokenClassification
from pathlib import Path
config = LayoutLMConfig(vocab_size=30522, hidden_size=256, num_hidden_layers=6,
    num_attention_heads=4, intermediate_size=768, max_position_embeddings=512,
    max_2d_position_embeddings=1024, num_labels=59)
student = LayoutLMForTokenClassification(config)
print('parameters:', sum(p.numel() for p in student.parameters()))


## Next training cell

The actual distillation trainer must be added after labels are present. It will combine
supervised token loss, teacher-logit distillation, row-boundary auxiliary loss, and GST
consistency checks. Do not evaluate on synthetic-only data.


In [ ]:
# Export smoke test only; run after the real model has been trained.
print('Student graph instantiated. Awaiting labeled data and teacher checkpoint.')
